# Where along the PT does injury act? Replication in independent spatial IRI data (notebook 105)

**The question.** Our positional AKI claims rest on two IR mice: acute injury rises along the PT toward S3
(notebooks 96, 99), transport loss is S3-weighted (notebook 94), and the S2→S3 gradient survives injury less than
the S1→S2 gradient (notebook 91). Two public Visium IRI datasets from independent labs have biological replicates at
our time points: Cheung 2022 (3 mice each at quiescent, 12 h and day 1) and Gharaie 2023 (2 control and 2 IRI at
24 h).

**The frozen protocol** is `docs/results/pt-external-spatial-iri.md` (commit 9f23ca4).
- Spots are deconvolved on Kirita 2020 nuclei; PT-dominant spots (PT weight ≥ 0.6) are placed on the PT axis by
  genes whose gradient survived injury in our mice.
- R1 injury rises along x; R2 transport loss is S3-weighted; R3 gradient survival per step; R4 notebook 91's
  location calls.

**Sections.** 1 setup; 2 reference and deconvolution; 3 position and programs; 4 R1–R2; 5 R3; 6 R4; 7 descriptive;
8 verdicts; 9 figures; 10 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import gzip
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
KIRITA = data_root / 'external' / 'kirita2020_iri'
CHEUNG = data_root / 'external' / 'cheung2022_iri_visium'
GHARAIE = data_root / 'external' / 'gharaie2023_iri_visium'
T76 = results_root / 'pt_package_aki' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
output = results_root / 'pt_external_spatial_iri'
tables, figures, cache_root = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '105.external_spatial.1'
SEED = 105
PT_TOP, PT_TOP_SENS, N_MARKERS, MIN_CP10K = .4, .25, 50, 1.0   # amendment 1: top share of each section by PT weight
SD_FLOOR, WIDTH, GRADED, PT_ENRICH = .25, .1, .5, 2.0
R_MARGIN, AGREE_MIN = .2, .6
BROAD = {'PTS1': 'PTS1', 'PTS2': 'PTS2', 'PTS3': 'PTS3', 'NewPT1': 'NewPT', 'NewPT2': 'NewPT', 'DTL-ATL': 'DTL-ATL',
         'MTAL': 'TAL', 'CTAL1': 'TAL', 'CTAL2': 'TAL', 'MD': 'TAL', 'DCT': 'DCT', 'DCT-CNT': 'DCT', 'CNT': 'CNT',
         'PC1': 'PC', 'PC2': 'PC', 'ICA': 'IC', 'ICB': 'IC', 'Pod': 'Pod', 'PEC': 'PEC', 'EC1': 'EC', 'EC2': 'EC',
         'Fib': 'Stroma', 'Per': 'Stroma', 'Mø': 'Immune', 'Tcell': 'Immune', 'Uro': 'Uro'}
TYPES = list(dict.fromkeys(BROAD.values()))
PT_TYPES = ['PTS1', 'PTS2', 'PTS3', 'NewPT']
KIRITA_FILES = {'Control': 'control', '12hours': '12hours', '2days': '2days'}
CHEUNG_SECTIONS = {'GSM6009058_visium1qui': ('control', 'quiescent'), 'GSM6009059_visium2qui': ('control', 'quiescent'),
                   'GSM6009060_visium3qui': ('control', 'quiescent'), 'GSM6009061_visium1d0p5': ('IR', '12 h'),
                   'GSM6009062_visium2d0p5': ('IR', '12 h'), 'GSM6009063_visium3d0p5': ('IR', '12 h'),
                   'GSM6009064_visium1d1': ('IR', 'day 1'), 'GSM6009065_visium2d1': ('IR', 'day 1'),
                   'GSM6009066_visium3d1': ('IR', 'day 1')}
GHARAIE_SECTIONS = {'GSM7429786_NL1': ('control', 'control'), 'GSM7429787_NL2': ('control', 'control'),
                    'GSM7429788_IL1': ('IR', '24 h'), 'GSM7429789_IL2': ('IR', '24 h')}
TESTS = [('Cheung', '12 h'), ('Cheung', 'day 1'), ('Gharaie', '24 h')]
REQUIRED = [KIRITA / 'GSE139107_MouseIRI.metadata.txt.gz', *[KIRITA / f'GSE139107_MouseIRI_{s}.dge.txt.gz' for s in KIRITA_FILES.values()],
            *[CHEUNG / f'{s}_filtered_feature_bc_matrix.h5' for s in CHEUNG_SECTIONS],
            *[GHARAIE / f'{s}_{suf}' for s in GHARAIE_SECTIONS for suf in ('matrix.mtx.gz', 'barcodes.tsv.gz', 'features.tsv.gz', 'tissue_positions_list.csv.gz')],
            T76 / 'positive_control_markers.csv', T76 / 'programs_transporters.csv', T76 / 'gene_sets.json', T76 / 'universe.csv',
            T90 / 'd_end_contrasts.csv.gz', T91 / 'i_genes.csv.gz']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.io import mmread
from scipy.stats import binomtest, spearmanr
from IPython.display import display

from pseudospace import external_spatial as es
from pseudospace.external_replication import dge_group_sums
from pseudospace.stage_cache import cached_payload
from pseudospace.tissue_location import position_expectation, program_score

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
COND_COLOR = {'control': '#0072B2', 'IR': '#D55E00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []

### 1.1 · The Visium sections (not cached)

G1: every Cheung barcode maps onto the Visium v1 grid (Gharaie's full position list).

In [ ]:
GRID = pd.read_csv(GHARAIE / 'GSM7429786_NL1_tissue_positions_list.csv.gz', header=None,
                   names=['barcode', 'in_tissue', 'row', 'col', 'px_row', 'px_col']).set_index('barcode')
assert len(GRID) == 4992


def unique_genes(names, X):
    """Collapse duplicated gene symbols by summing their columns (spots x genes CSR)."""
    names = pd.Index(names)
    if names.is_unique:
        return names, X
    codes, uniq = pd.factorize(names)
    agg = sparse.csr_matrix((np.ones(len(codes)), (np.arange(len(codes)), codes)), shape=(len(codes), len(uniq)))
    return pd.Index(uniq), (X @ agg).tocsr()


SECTIONS = {}
for name, (cond, tp) in CHEUNG_SECTIONS.items():
    with h5py.File(CHEUNG / f'{name}_filtered_feature_bc_matrix.h5', 'r') as f:
        m = f['matrix']
        X = sparse.csc_matrix((m['data'][:], m['indices'][:], m['indptr'][:]), shape=tuple(m['shape'][:])).T.tocsr()
        barcodes = m['barcodes'][:].astype(str)
        genes = m['features']['name'][:].astype(str)
    hit = GRID.index.get_indexer(barcodes)
    assert (hit >= 0).all(), f'G1: {name} has barcodes off the v1 grid'
    genes, X = unique_genes(genes, X)
    rows, cols = GRID.row.to_numpy()[hit], GRID.col.to_numpy()[hit]
    SECTIONS[name] = {'dataset': 'Cheung', 'condition': cond, 'time': tp, 'X': X, 'genes': genes, 'rows': rows, 'cols': cols,
                      'depth': es.edge_distance(rows, cols, GRID.row.to_numpy(), GRID.col.to_numpy())}
for name, (cond, tp) in GHARAIE_SECTIONS.items():
    X = sparse.csr_matrix(mmread(GHARAIE / f'{name}_matrix.mtx.gz').T)
    barcodes = pd.read_csv(GHARAIE / f'{name}_barcodes.tsv.gz', header=None)[0].astype(str).to_numpy()
    genes = pd.read_csv(GHARAIE / f'{name}_features.tsv.gz', header=None, sep='\t')[1].astype(str).to_numpy()
    pos = pd.read_csv(GHARAIE / f'{name}_tissue_positions_list.csv.gz', header=None,
                      names=['barcode', 'in_tissue', 'row', 'col', 'px_row', 'px_col']).set_index('barcode')
    hit = pos.index.get_indexer(barcodes)
    assert (hit >= 0).all(), f'{name}: barcodes without positions'
    genes, X = unique_genes(genes, X)
    rows, cols = pos.row.to_numpy()[hit], pos.col.to_numpy()[hit]
    SECTIONS[name] = {'dataset': 'Gharaie', 'condition': cond, 'time': tp, 'X': X, 'genes': genes, 'rows': rows, 'cols': cols,
                      'depth': es.edge_distance(rows, cols, pos.row.to_numpy(), pos.col.to_numpy())}
for name, s in SECTIONS.items():
    umi = np.asarray(s['X'].sum(axis=1)).ravel()
    print(f"{name}: {s['dataset']} {s['condition']} {s['time']} · {s['X'].shape[0]:,} spots · median UMI {np.median(umi):,.0f}")
guard_log.append('G1: every Cheung barcode maps onto the Visium v1 grid; Gharaie positions are deposited')

## 2 · Reference profiles (Kirita 2020, cached) and deconvolution

In [ ]:
meta = pd.read_csv(KIRITA / 'GSE139107_MouseIRI.metadata.txt.gz', sep='\t', index_col=0)
meta = meta[meta.Group.isin(KIRITA_FILES) & meta.celltype.isin(BROAD)]


def build_reference():
    frames = []
    for group, stem in KIRITA_FILES.items():
        sub = meta[meta.Group.eq(group)]
        labels = dict(zip(sub.index, group + '|' + sub.celltype.map(BROAD)))
        genes, sums, _ = dge_group_sums(KIRITA / f'GSE139107_MouseIRI_{stem}.dge.txt.gz', labels)
        frames.append(sums)
    table = pd.concat(frames, axis=1).fillna(0)
    return {'genes': table.index.astype(str).to_numpy().astype('U'), 'columns': table.columns.astype(str).to_numpy().astype('U'),
            'sums': table.to_numpy(np.int64)}


t0 = time.perf_counter()
REFP = cached_payload('kirita_reference', build_reference, root=cache_root,
                      params={'logic': NOTEBOOK_LOGIC_VERSION, 'broad': BROAD, 'files': KIRITA_FILES}, code=dge_group_sums)
SUMS = pd.DataFrame(REFP['sums'], index=REFP['genes'].astype(str), columns=REFP['columns'].astype(str))
SUMS = SUMS.groupby(level=0).sum()
print(f'Kirita reference: {SUMS.shape[0]:,} genes × {SUMS.shape[1]} time|type columns ({time.perf_counter() - t0:.0f} s)')
BY_TYPE = SUMS.T.groupby(SUMS.columns.str.split('|').str[1]).sum().T.reindex(columns=TYPES)
assert BY_TYPE.notna().all().all() and (BY_TYPE.sum() > 0).all(), 'G2: a broad type is missing from the reference'
PROFILE = BY_TYPE / BY_TYPE.sum() * 1e4
LATE_SUMS = SUMS[[c for c in SUMS.columns if c.split('|')[0] in ('12hours', '2days')]]
LATE = LATE_SUMS.T.groupby(LATE_SUMS.columns.str.split('|').str[1]).sum().T
LATE = LATE / LATE.sum() * 1e4
guard_log.append(f'G2: Kirita reference with {len(TYPES)} broad types from the control, 12 h and 2 d files')

In [ ]:
def cp10k(X):
    lib = np.asarray(X.sum(axis=1)).ravel().astype(float)
    return sparse.diags(1e4 / np.maximum(lib, 1)) @ X


WEIGHTS = {}
marker_rows = []
for dataset in ('Cheung', 'Gharaie'):
    names = [n for n, s in SECTIONS.items() if s['dataset'] == dataset]
    shared = PROFILE.index.intersection(SECTIONS[names[0]]['genes'])
    for n in names[1:]:
        shared = shared.intersection(SECTIONS[n]['genes'])
    prof = PROFILE.loc[shared]
    prof = prof / prof.sum() * 1e4
    markers = es.type_markers(prof, n=N_MARKERS, min_cp10k=MIN_CP10K)
    union = pd.Index(sorted(set().union(*markers.values())))
    marker_rows += [{'dataset': dataset, 'type': t, 'gene': g} for t, gs in markers.items() for g in gs]
    for n in names:
        s = SECTIONS[n]
        spots = cp10k(s['X'])[:, s['genes'].get_indexer(union)].toarray()
        WEIGHTS[n] = pd.DataFrame(es.nnls_weights(spots, prof.loc[union].to_numpy()), columns=prof.columns)
pd.DataFrame(marker_rows).to_csv(tables / 'markers.csv', index=False)
for n, w in WEIGHTS.items():
    pt = w[PT_TYPES].sum(axis=1).to_numpy()
    SECTIONS[n]['pt'] = pt
    SECTIONS[n]['pt rank'] = es.rank_position(pt, np.zeros(len(pt)))    # within-section rank in (0, 1)
display(pd.DataFrame({n: {'spots': len(w), 'PT weight ≥ 0.6 (frozen rule)': int((SECTIONS[n]['pt'] >= .6).sum()),
                          'PT-dominant (top 40%)': int((SECTIONS[n]['pt rank'] > 1 - PT_TOP).sum()),
                          'median PT weight of the top 40%': float(np.median(SECTIONS[n]['pt'][SECTIONS[n]['pt rank'] > 1 - PT_TOP])),
                          **{f'mean w {t}': float(w[t].mean()) for t in TYPES}} for n, w in WEIGHTS.items()}).T.round(3))

## 3 · Positional genes, position and program scores

In [ ]:
END90 = pd.read_csv(T90 / 'd_end_contrasts.csv.gz').set_index('mouse')
END90.index = END90.index.astype(str)
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
kept = G91.index[G91.survival.eq('kept')]
e1, e2 = END90['E Ctrl1A2'], END90['E Ctrl1A4']
pt_cp = PROFILE[['PTS1', 'PTS2', 'PTS3']].mean(axis=1)
non_pt = PROFILE.drop(columns=PT_TYPES).max(axis=1)
pt_enriched = PROFILE.index[(pt_cp >= PT_ENRICH * non_pt).to_numpy()]
EARLY = kept.intersection(END90.index[(e1 <= -GRADED) & (e2 <= -GRADED)]).intersection(pt_enriched)
LATE_G = kept.intersection(END90.index[(e1 >= GRADED) & (e2 >= GRADED)]).intersection(pt_enriched)
pd.concat([pd.DataFrame({'gene': EARLY, 'side': 'early'}), pd.DataFrame({'gene': LATE_G, 'side': 'late'})]).to_csv(
    tables / 'positional_genes.csv', index=False)
print(f'Positional genes: {len(EARLY)} early, {len(LATE_G)} late')
guard_log.append(f'G4: positional genes {len(EARLY)} early and {len(LATE_G)} late (kept in 91, graded in both controls, PT-enriched in Kirita)')

markers76 = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
cyc = sorted((set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(sets76['MSigDB_Hallmark_2020::E2F Targets'])) & u76)
tr = pd.read_csv(T76 / 'programs_transporters.csv')
tpt = sorted(tr.mouse[tr.in_T_PT.astype(bool)].astype(str))
assert (len(markers76), len(tpt), len(cyc)) == (18, 93, 297), 'G3: program sizes'
inj_pt = [g for g in markers76 if g in LATE.index and LATE.loc[g, 'NewPT'] > LATE.loc[g, 'TAL']]
PROGRAMS = {'INJ': markers76, 'INJ-PT': inj_pt, 'TPT': tpt, 'CYC': cyc}
print('INJ-PT:', ', '.join(inj_pt))

spot_frames = []
for dataset in ('Cheung', 'Gharaie'):
    names = [n for n, s in SECTIONS.items() if s['dataset'] == dataset]
    genes = SECTIONS[names[0]]['genes']
    for n in names[1:]:
        genes = genes.intersection(SECTIONS[n]['genes'])
    blocks, frames = [], []
    for n in names:
        s = SECTIONS[n]
        keep = s['pt rank'] > 1 - PT_TOP
        expr = cp10k(s['X'])[np.flatnonzero(keep)][:, s['genes'].get_indexer(genes)].tocsr()
        expr.data = np.log1p(expr.data)
        blocks.append(expr)
        w = WEIGHTS[n][keep.astype(bool)].reset_index(drop=True)
        seg = (0.5 * w.PTS2 + 1.0 * w.PTS3) / (w.PTS1 + w.PTS2 + w.PTS3).replace(0, np.nan)
        frames.append(pd.DataFrame({'dataset': dataset, 'section': n, 'condition': s['condition'], 'time': s['time'],
                                    'row': s['rows'][keep], 'col': s['cols'][keep], 'depth': s['depth'][keep],
                                    'pt': s['pt'][keep], 'pt rank': s['pt rank'][keep], 'x_seg raw': seg.to_numpy(),
                                    **{f'w {t}': w[t].to_numpy() for t in TYPES}}))
    E = sparse.vstack(blocks).tocsr()
    F = pd.concat(frames, ignore_index=True)
    ref = F.condition.eq('control').to_numpy()
    cover = {}
    for name, gs in {'early': EARLY, 'late': LATE_G, **PROGRAMS}.items():
        present = [g for g in gs if g in genes]
        cover[name] = f'{len(present)}/{len(gs)}'
        F[f'score {name}'] = program_score(E[:, genes.get_indexer(present)], ref, floor=SD_FLOOR)
    print(dataset, 'gene coverage:', cover)
    F['positional score'] = F['score late'] - F['score early']
    F['x'] = es.rank_position(F['positional score'], F.section)
    F['x_seg'] = es.rank_position(F['x_seg raw'].fillna(F['x_seg raw'].median()), F.section)
    controls = sorted(F.section[F.condition.eq('control')].unique())
    for name in PROGRAMS:
        F[f'e {name}'] = F[f'score {name}'] - position_expectation(F.x, F[f'score {name}'], F.section, controls, width=WIDTH, cross_fit=True)
    spot_frames.append((dataset, F, E, genes))
SPOTS = pd.concat([f for _, f, _, _ in spot_frames], ignore_index=True)
SPOTS.drop(columns=[c for c in SPOTS.columns if c.startswith('score early') or c.startswith('score late')]).to_csv(tables / 'spots.csv', index=False)
G5 = SPOTS[SPOTS.condition.eq('control')].groupby('section').apply(
    lambda g: pd.Series({'ρ(x, x_seg)': spearmanr(g.x, g.x_seg).statistic, 'ρ(x, depth)': spearmanr(g.x, g.depth).statistic}))
display(G5.round(3))
guard_log.append('G5 (reported): in control sections ρ(x, x_seg) ' + ', '.join(f'{v:.2f}' for v in G5['ρ(x, x_seg)']))

## 4 · R1 and R2: injury and transport loss along the axis

In [ ]:
def section_rho(frame, col):
    return frame.groupby('section').apply(lambda g: spearmanr(g[col], g.x).statistic)


rho_rows = []
for dataset, tp in TESTS:
    F = SPOTS[SPOTS.dataset.eq(dataset)]
    for col in ('e INJ', 'e INJ-PT', 'e TPT', 'e CYC'):
        rho = section_rho(F, col)
        for sec, v in rho.items():
            cond = F.loc[F.section.eq(sec), 'condition'].iloc[0]
            t = F.loc[F.section.eq(sec), 'time'].iloc[0]
            if cond == 'control' or t == tp:
                rho_rows.append({'dataset': dataset, 'test': tp, 'program': col, 'section': sec, 'condition': cond, 'rho': v})
RHO = pd.DataFrame(rho_rows)


def verdict(sub, sign):
    ir, ctrl = sub[sub.condition.eq('IR')].rho, sub[sub.condition.eq('control')].rho
    diff = ir.mean() - ctrl.mean()
    if sign > 0:
        ok = bool(ir.min() > ctrl.max() and diff >= R_MARGIN)
    else:
        ok = bool(ir.max() < ctrl.min() and diff <= -R_MARGIN)
    return {'IR ρ': ', '.join(f'{v:.2f}' for v in ir), 'control ρ': ', '.join(f'{v:.2f}' for v in ctrl),
            'IR − control': diff, 'replicates': ok}


R12 = pd.DataFrame([{'dataset': d, 'test': tp, 'reading': r, 'program': p, **verdict(RHO[(RHO.dataset == d) & (RHO.test == tp) & (RHO.program == p)], sgn)}
                    for d, tp in TESTS for r, p, sgn in (('R1', 'e INJ', 1), ('R1 (INJ-PT)', 'e INJ-PT', 1),
                                                       ('R2', 'e TPT', -1), ('CYC (descriptive)', 'e CYC', 1))])
R12.to_csv(tables / 'r1_r2.csv', index=False)
display(R12.round(3))

# sensitivity: PT weight ≥ 0.75 and the deconvolution position
sens_rows = []
for dataset, tp in TESTS:
    F = SPOTS[SPOTS.dataset.eq(dataset) & (SPOTS.condition.eq('control') | SPOTS.time.eq(tp))]
    for label, sub, xcol in (('PT top 25%', F[F['pt rank'] > 1 - PT_TOP_SENS], 'x'), ('x from deconvolution', F, 'x_seg')):
        for col, sgn in (('e INJ', 1), ('e TPT', -1)):
            r = sub.groupby('section').apply(lambda g: spearmanr(g[col], g[xcol]).statistic)
            cond = sub.groupby('section').condition.first()
            frame = pd.DataFrame({'rho': r, 'condition': cond})
            sens_rows.append({'dataset': dataset, 'test': tp, 'sensitivity': label, 'program': col, **verdict(frame, sgn)})
R12_SENS = pd.DataFrame(sens_rows)
R12_SENS.to_csv(tables / 'r1_r2_sensitivity.csv', index=False)
display(R12_SENS.round(3))

## 5 · R3: gradient survival per step

In [ ]:
r3_rows = []
for dataset, F, E, genes in spot_frames:
    third = np.minimum((F.x.to_numpy() * 3).astype(int), 2)
    means = {}
    for sec in F.section.unique():
        m = F.section.eq(sec).to_numpy()
        means[sec] = np.vstack([np.asarray(E[np.flatnonzero(m & (third == k))].mean(axis=0)).ravel() for k in range(3)])
    controls = [s for s in means if SECTIONS[s]['condition'] == 'control']
    steps = {'S1→S2': (0, 1), 'S2→S3': (1, 2), 'whole PT': (0, 2)}
    for tp in [t for d, t in TESTS if d == dataset]:
        irs = [s for s in means if SECTIONS[s]['time'] == tp]
        for step, (a, b) in steps.items():
            c = np.vstack([means[s][b] - means[s][a] for s in controls])
            graded = (np.abs(c) >= GRADED).all(axis=0) & ((np.sign(c) == np.sign(c[0])).all(axis=0))
            ref = c[:, graded].mean(axis=0)
            for s in irs:
                y = (means[s][b] - means[s][a])[graded]
                r3_rows.append({'dataset': dataset, 'test': tp, 'section': s, 'step': step, 'graded genes': int(graded.sum()),
                                'survival': es.origin_slope(y, ref)})
R3 = pd.DataFrame(r3_rows)
R3_WIDE = R3.pivot_table(index=['dataset', 'test', 'section'], columns='step', values='survival')
R3_WIDE['S2→S3 < S1→S2'] = R3_WIDE['S2→S3'] < R3_WIDE['S1→S2']
R3.to_csv(tables / 'r3_survival.csv', index=False)
display(R3.pivot_table(index=['dataset', 'test'], columns='step', values='graded genes', aggfunc='first'))
display(R3_WIDE.round(3))
MIN_GRADED = 10
testable = R3.groupby(['dataset', 'test'])['graded genes'].min() >= MIN_GRADED
R3_VERDICT = R3_WIDE.groupby(['dataset', 'test'])['S2→S3 < S1→S2'].all().astype(object)
R3_VERDICT[~testable.reindex(R3_VERDICT.index).to_numpy()] = 'not testable'

## 6 · R4: notebook 91's location calls

In [ ]:
calls = G91[G91['location change'].astype(bool)]
r4_rows = []
for dataset, F, E, genes in spot_frames:
    late = F.x.to_numpy() >= 2 / 3
    sec_late, sec_all = {}, {}
    for sec in F.section.unique():
        m = F.section.eq(sec).to_numpy()
        sec_late[sec] = np.asarray(E[np.flatnonzero(m & late)].mean(axis=0)).ravel()
        sec_all[sec] = np.asarray(E[np.flatnonzero(m)].mean(axis=0)).ravel()
    controls = [s for s in sec_late if SECTIONS[s]['condition'] == 'control']
    use = calls.index.intersection(genes)
    idx = genes.get_indexer(use)
    for tp in [t for d, t in TESTS if d == dataset]:
        irs = [s for s in sec_late if SECTIONS[s]['time'] == tp]
        change = (np.mean([sec_late[s] for s in irs], axis=0) - np.mean([sec_late[s] for s in controls], axis=0)
                  - (np.mean([sec_all[s] for s in irs], axis=0) - np.mean([sec_all[s] for s in controls], axis=0)))[idx]
        sign = np.sign(calls.loc[use, 'Δ late'].to_numpy())
        ok = (change != 0) & (sign != 0)
        agree = int((np.sign(change[ok]) == sign[ok]).sum())
        p = binomtest(agree, int(ok.sum()), .5, alternative='greater').pvalue
        r4_rows.append({'dataset': dataset, 'test': tp, 'calls': int(ok.sum()), 'agree': agree, 'share': agree / ok.sum(), 'p': p,
                        'replicates': bool(agree / ok.sum() >= AGREE_MIN and p <= .05)})
R4 = pd.DataFrame(r4_rows)
R4.to_csv(tables / 'r4_location.csv', index=False)
display(R4.round(4))

## 7 · Descriptive profiles along the axis

In [ ]:
SPOTS['xbin'] = np.minimum((SPOTS.x * 10).astype(int), 9)
PROFILES = SPOTS.groupby(['dataset', 'time', 'condition', 'section', 'xbin'])[['e INJ', 'e INJ-PT', 'e TPT', 'e CYC', 'w NewPT', 'depth']].mean()
PROFILES.to_csv(tables / 'profiles.csv')
display(PROFILES.groupby(['dataset', 'time', 'xbin'])['e INJ'].mean().unstack('xbin').round(2))
DEPTH = SPOTS.groupby('section').apply(lambda g: spearmanr(g.x, g.depth).statistic)
print('ρ(x, depth) per section:', DEPTH.round(2).to_dict())

## 8 · Verdicts

In [ ]:
r1 = R12[R12.reading.eq('R1')].set_index(['dataset', 'test']).replicates
OVERALL = bool(r1[('Cheung', '12 h')] and (r1[('Cheung', 'day 1')] or r1[('Gharaie', '24 h')]))
VERDICTS = pd.DataFrame([
    *[(f'R1 injury rises along x · {d} {t}', str(r1[(d, t)])) for d, t in TESTS],
    *[(f'R1 (INJ-PT) · {d} {t}', str(v)) for (d, t), v in R12[R12.reading.eq('R1 (INJ-PT)')].set_index(['dataset', 'test']).replicates.items()],
    *[(f'R2 transport loss S3-weighted · {d} {t}', str(v)) for (d, t), v in R12[R12.reading.eq('R2')].set_index(['dataset', 'test']).replicates.items()],
    *[(f'R3 S2→S3 survival below S1→S2 · {d} {t}', str(v)) for (d, t), v in R3_VERDICT.items()],
    *[(f'R4 location calls · {r.dataset} {r.test}', f'{r.share:.2f} agree (p {r.p:.3g}): {r.replicates}') for r in R4.itertuples()],
    ('Overall: positional injury replicates externally', str(OVERALL)),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 9 · Figures (plots only)

In [ ]:
ctrl_name, ir_name = 'GSM6009059_visium2qui', 'GSM6009062_visium2d0p5'
fig, axes = plt.subplots(2, 2, figsize=(120 * MM, 110 * MM))
for col, name in enumerate((ctrl_name, ir_name)):
    s = SECTIONS[name]
    xs, ys = es.visium_v1_um(s['rows'], s['cols'])
    sub = SPOTS[SPOTS.section.eq(name)]
    px, py = es.visium_v1_um(sub.row, sub.col)
    for row, (val, cmap, lim) in enumerate((('x', 'viridis', (0, 1)), ('e INJ', 'RdBu_r', (-2, 2)))):
        ax = axes[row, col]
        ax.scatter(xs, -ys, s=1, color=LIGHT, lw=0)
        ax.scatter(px, -py, s=1.5, c=sub[val], cmap=cmap, vmin=lim[0], vmax=lim[1], lw=0)
        ax.set_aspect('equal'); ax.axis('off')
        if row == 0:
            ax.set_title(f"{s['condition']} · {s['time']}")
save(fig, '105a_section_maps')
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(150 * MM, 75 * MM), sharex=True)
mids = (np.arange(10) + .5) / 10
for c, (dataset, tp) in enumerate(TESTS):
    for r, col in enumerate(('e INJ', 'e TPT')):
        ax = axes[r, c]
        prof = PROFILES.reset_index()
        prof = prof[prof.dataset.eq(dataset) & (prof.condition.eq('control') | prof.time.eq(tp))]
        for sec, g in prof.groupby('section'):
            ax.plot(mids[g.xbin], g[col], color=COND_COLOR[g.condition.iloc[0]], lw=.8)
        ax.axhline(0, color=LIGHT, lw=.5)
        if r == 0:
            ax.set_title(f'{dataset} · {tp}')
        if r == 1:
            ax.set_xlabel('PT position')
    axes[0, 0].set_ylabel('injury residual'); axes[1, 0].set_ylabel('transport residual')
save(fig, '105b_profiles')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 45 * MM), sharey=True)
for ax, prog in zip(axes, ('e INJ', 'e TPT')):
    sub = RHO[RHO.program.eq(prog)]
    for i, (d, t) in enumerate(TESTS):
        g = sub[(sub.dataset == d) & (sub.test == t)]
        for cond, off in (('control', -.12), ('IR', .12)):
            v = g[g.condition.eq(cond)].rho
            ax.scatter(np.full(len(v), i + off), v, color=COND_COLOR[cond], s=10)
    ax.axhline(0, color=LIGHT, lw=.5)
    ax.set_xticks(range(len(TESTS)), [f'{d}\n{t}' for d, t in TESTS])
axes[0].set_ylabel('Spearman ρ with PT position')
save(fig, '105c_rho_per_section')
plt.show()

fig, ax = plt.subplots(figsize=(80 * MM, 45 * MM))
steps = ['S1→S2', 'S2→S3', 'whole PT']
for i, (d, t) in enumerate(TESTS):
    g = R3[(R3.dataset == d) & (R3.test == t)]
    for j, st in enumerate(steps):
        v = g[g.step.eq(st)].survival
        ax.scatter(np.full(len(v), i * 4 + j), v, color=['#56B4E9', '#D55E00', MUTED][j], s=10)
ax.axhline(1, color=LIGHT, lw=.5); ax.axhline(0, color=LIGHT, lw=.5)
ax.set_xticks([i * 4 + 1 for i in range(len(TESTS))], [f'{d}\n{t}' for d, t in TESTS])
ax.set_ylabel('gradient survival')
save(fig, '105d_survival')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Duplicated gene symbols within a section are summed; each dataset is analysed on the genes shared by all its sections.'),
    ('reading', 'Deconvolution markers are chosen on the Kirita genes shared with the dataset, with the profiles renormalised to CP10k on those genes.'),
    ('reading', 'R1–R2 compare each time point\'s IR sections with the dataset\'s control sections (Cheung: the 3 quiescent sections for both 12 h and day 1).'),
    ('reading', 'The deconvolution position x_seg of a spot with no segment weight takes the section median before ranking.'),
    ('reading', 'R4 uses notebook 91\'s location calls present in the dataset; a call with zero change is left out of the binomial test.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '105_pt_external_spatial_iri', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'inj_pt': inj_pt, 'r1_r2': R12.round(4).to_dict('records'), 'r1_r2_sensitivity': R12_SENS.round(4).to_dict('records'),
       'r3': R3_WIDE.round(4).reset_index().to_dict('records'), 'r4': R4.round(4).to_dict('records'),
       'depth_rho': DEPTH.round(3).to_dict(), 'overall': OVERALL, 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('overall', 'seconds')}, indent=2, default=str))